##Widgets

In [0]:
%sql
/*
--bronze
drop table IF EXISTS etl_vinkos.bronze.archivos_a_procesar;
drop table IF EXISTS etl_vinkos.bronze.estadistica_contenedor;


--Silver
drop table IF EXISTS etl_vinkos.bronze.visitantes_agregados;

DROP SCHEMA IF EXISTS etl_vinkos.raw CASCADE;
DROP SCHEMA IF EXISTS etl_vinkos.bronze CASCADE;
DROP SCHEMA IF EXISTS etl_vinkos.silver CASCADE;
DROP SCHEMA IF EXISTS etl_vinkos.gold CASCADE;

DROP CATALOG IF EXISTS catalogo CASCADE;

DROP EXTERNAL LOCATION IF EXISTS "exlt-backup";
DROP EXTERNAL LOCATION IF EXISTS "exlt-raw";
DROP EXTERNAL LOCATION IF EXISTS "exlt-bronze";
DROP EXTERNAL LOCATION IF EXISTS "exlt-silver";
DROP EXTERNAL LOCATION IF EXISTS "exlt-gold";
--DROP EXTERNAL LOCATION IF EXISTS `exlt-metastore`;
*/

In [0]:
'''

dbutils.fs.rm(
"abfss://bronze@adlsvinkos.dfs.core.windows.net/",
True
)

dbutils.fs.rm(
"abfss://silver@adlsvinkos.dfs.core.windows.net/",
True
)

dbutils.fs.rm(
"abfss://gold@adlsvinkos.dfs.core.windows.net/",
True
)

dbutils.fs.rm(
"abfss://backup@adlsvinkos.dfs.core.windows.net/",
True
)
'''

In [0]:
%sql
/*SHOW CATALOGS;

SHOW SCHEMAS IN etl_vinkos;

SHOW TABLES IN etl_vinkos.bronze;

SHOW EXTERNAL LOCATIONS;*/

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
--- crear widgets
create widget text storageName default "adlsvinkosetl";
create widget text catalogo default "etl_vinkos";
create widget text container default "raw";
create widget text schemaBronze default "bronze";
create widget text schemaSilver default "silver";
create widget text schemaGold default "gold";
create widget text credential_ default "etlcredential";


In [0]:
%python
storageName = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("container")
schemaBronze = dbutils.widgets.get("schemaBronze")
schemaSilver = dbutils.widgets.get("schemaSilver")
schemaGold = dbutils.widgets.get("schemaGold")
catalogo = dbutils.widgets.get("catalogo")
credential_ = dbutils.widgets.get("credential_")

# guardar el widget en una variable

##External Locations

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-backup`
URL 'abfss://backup@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-gold`
URL 'abfss://gold@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credential)
COMMENT 'Ubicación externa para las tablas delta del Data deltaake';

##Catalogs y Schemas

In [0]:
%sql
--- crear el catalogo

CREATE CATALOG IF NOT EXISTS ${catalogo}
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';
     

In [0]:

%sql
---  crear los eschemas y volume necesario

CREATE SCHEMA IF NOT EXISTS ${catalogo}.${container};
CREATE SCHEMA IF NOT EXISTS ${catalogo}.${schemaBronze};
CREATE SCHEMA IF NOT EXISTS ${catalogo}.${schemaSilver};
CREATE SCHEMA IF NOT EXISTS ${catalogo}.${schemaGold};


--DROP CATALOG ${catalogo} CASCADE;

--CREATE VOLUME IF NOT EXISTS etl_vinkos.container.datasets;

##Tablas Bronze


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaBronze}.estadistica_contenedor (
email           String,
jyv             String,
badmail	        String,
baja            String,
fecha_envio     String,
fecha_open      String,
opens           String, 
opens_virales   String,
fecha_click     String,
clicks          String,
clicks_virales  String,
links           String,
ips             String,
navegadores     String,
plataformas     String,
archivo_origen  String,
fecha_carga     Timestamp
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/estadistica_contenedor';




##Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  ${catalogo}.${schemaSilver}.archivos_a_procesar (

nom_archivo       STRING,
fecha_proceso     TIMESTAMP,
total_registros   INT,
estatus           STRING,      
mensaje_error     STRING       

)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/archivos_a_procesar';

##Tablas Gold

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  ${catalogo}.${schemaGold}.estadistica_log_error (
id_error BIGINT GENERATED ALWAYS AS IDENTITY,
email           String,
jyv             String,
badmail	        String,
baja            String,
fecha_envio     String,
fecha_open      String,
opens           String, 
opens_virales   String,
fecha_click     String,
clicks          String,
clicks_virales  String,
links           String,
ips             String,
navegadores     String,
plataformas     String,
archivo_origen  String,
fecha_carga     Timestamp,
nom_archivo     String,
error           STRING
)
USING DELTA
--PARTITIONED BY (order_id)
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/estadistica_log_error";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  ${catalogo}.${schemaGold}.estadistica (
id_estadistica BIGINT GENERATED ALWAYS AS IDENTITY,
email           String,
jyv             String,
badmail	        String,
baja            String,
fecha_envio     Timestamp,
fecha_open      Timestamp,
opens           integer, 
opens_virales   integer,
fecha_click     Timestamp,
clicks          integer,
clicks_virales  integer,
links           String,
ips             String,
navegadores     String,
plataformas     String,
archivo_origen  String,
fecha_carga     Timestamp,
nom_archivo     String
)
USING DELTA
--PARTITIONED BY (order_id)
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/estadistica";

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.${schemaGold}.visitantes (
id_visitante BIGINT GENERATED ALWAYS AS IDENTITY,
email String,
fechaPrimeraVisita Timestamp,
fechaUltimaVisita Timestamp,
visitasTotales Integer,
visitasAnioActual Integer,
visitasMesActual Integer,
fechaCarga Timestamp,
fechaActualizacion Timestamp
)
USING DELTA
--PARTITIONED BY (order_id)
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/visitantes"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS   ${catalogo}.${schemaGold}.bitacora_ctrl_file (
id_bitacora BIGINT GENERATED ALWAYS AS IDENTITY,
nom_archivo       STRING,
fecha_proceso     TIMESTAMP,
total_registros   INT,
estatus           STRING,      
mensaje_error     STRING       

)
USING DELTA
LOCATION 'abfss://gold@${storageName}.dfs.core.windows.net/bitacora_ctrl_file';
